# Big West Championships | 2021–2026
Parse locally saved TFRRS championship HTML into separate running and field CSVs. Manually approved field MARK columns are preserved. Run all cells from top to bottom.

In [7]:
from pathlib import Path
from io import StringIO
import re
import pandas as pd
from bs4 import BeautifulSoup

RAW_DIR = Path('../data/raw/championships')
OUTPUT_DIR = Path('../data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
YEARS = range(2021, 2027)
GENDERS = ('Men', 'Women')
# Strict mode: do not silently skip years if a file is missing.
STRICT_INPUTS = False  # Set True to require every year/gender HTML file


In [8]:
FIELD_KEYWORDS = ('long jump', 'triple jump', 'high jump', 'pole vault',
                  'shot put', 'discus', 'hammer', 'javelin')
COMBINED_KEYWORDS = ('heptathlon', 'decathlon', 'pentathlon')

def get_event_info(table_tag):
    heading = table_tag.find_previous(['h1', 'h2', 'h3', 'h4', 'h5'])
    while heading:
        text = heading.get_text(' ', strip=True)
        if re.search(r"(?:Women['’]s|Men['’]s)", text, re.I):
            break
        heading = heading.find_previous(['h1', 'h2', 'h3', 'h4', 'h5'])
    if not heading:
        return None, None, None, None
    text = heading.get_text(' ', strip=True)
    event = re.split(r'\b(?:Finals?|Preliminaries|Semi-Finals?|Flight|Heat|Section)\b',
                     text, maxsplit=1, flags=re.I)[0]
    event = re.sub(r"^(?:Women['’]s|Men['’]s)\s+", '', event).strip()
    low = event.lower()
    event_type = ('combined' if any(x in low for x in COMBINED_KEYWORDS)
                  else 'field' if any(x in low for x in FIELD_KEYWORDS)
                  else 'running')
    round_name = ('Flight' if re.search(r'\bFlight\b', text, re.I)
                  else 'Heat' if re.search(r'\bHeat\b', text, re.I)
                  else 'Preliminaries' if re.search(r'\bPreliminaries\b', text, re.I)
                  else 'Finals' if re.search(r'\bFinals?\b', text, re.I)
                  else 'Other')
    return event, event_type, round_name, text

def is_field_summary_table(df):
    return {'PL', 'NAME', 'TEAM', 'MARK'}.issubset(df.columns)

def parse_time_seconds(value):
    """Convert sprint or mm:ss.xx race times into seconds."""
    if pd.isna(value):
        return float('nan')
    try:
        pieces = str(value).strip().split(':')
        return sum(float(piece) * 60**power
                   for power, piece in enumerate(reversed(pieces)))
    except (ValueError, TypeError):
        return float('nan')


def choose_running_time_column(df):
    """Find the TIME column that follows official finish-place ordering.

    Saved HTML may contain multiple TIME columns with spurious values.
    The real results should be nondecreasing by finish place. If the
    ordering cannot identify a unique column, fail rather than guess.
    """
    cols = [c for c in df.columns if re.fullmatch(r'TIME(?:\.\d+)?', str(c))]
    if not cols:
        return None
    if len(cols) == 1:
        return cols[0]
    places = pd.to_numeric(df['PL'], errors='coerce')
    scores = []
    for col in cols:
        seconds = df[col].map(parse_time_seconds)
        valid = places.notna() & seconds.notna()
        if valid.sum() < 3:
            continue
        ordered = pd.DataFrame({'place': places[valid], 'time': seconds[valid]}).sort_values('place')
        # Equal places can represent ties; ignore their relative order.
        diffs = ordered['time'].diff()
        place_diffs = ordered['place'].diff()
        violations = int(((diffs < -0.001) & (place_diffs > 0)).sum())
        scores.append((violations, col))
    if not scores:
        return None
    best = min(s[0] for s in scores)
    winners = [col for violations, col in scores if violations == best]
    if best != 0 or len(winners) != 1:
        return None
    return winners[0]


def get_official_mark_column(df, event_type, round_name, event=None, verified_marks=None):
    if event_type == 'field':
        cols = [c for c in df.columns if re.fullmatch(r'MARK(?:\.\d+)?', str(c))]
        if not cols:
            return None
        selected = (verified_marks or {}).get(event, cols[0] if len(cols) == 1 else None)
        if selected is not None and selected not in cols:
            raise ValueError(f'Approved column {selected} not found for {event}; candidates: {cols}')
        return selected
    return choose_running_time_column(df)

# Men's main field tables are discovered by event heading, not by women's table numbers.
def find_main_field_tables(tables, table_tags):
    found = {}
    for i, (table, tag) in enumerate(zip(tables, table_tags)):
        event, event_type, round_name, _ = get_event_info(tag)
        if (event_type == 'field' and round_name not in ('Flight', 'Heat', 'Preliminaries')
                and is_field_summary_table(table)):
            if event in found:
                raise ValueError(f'Multiple main field tables for {event}: {found[event]}, {i}')
            found[event] = i
    return found


def clean_event_table(table, tag, index, gender, year, field_tables, verified_marks):
    event, event_type, round_name, heading = get_event_info(tag)
    if not event or event_type == 'combined':
        return None
    df = table.copy()
    if event_type == 'field':
        # Use only the main overall results table, not the flight tables.
        if index != field_tables.get(event):
            return None
        if not is_field_summary_table(df):
            return None
    elif not {'PL', 'NAME', 'TEAM'}.issubset(df.columns):
        return None

    mark_col = get_official_mark_column(df, event_type, round_name, event, verified_marks)
    if mark_col is None:
        return None
    df = df.rename(columns={'PL':'place', 'NAME':'athlete_name', 'YEAR':'class_year',
                            'TEAM':'school', 'SC':'points', 'WIND':'wind',
                            'CONV':'converted_mark', mark_col:'mark'})
    ids = []
    for tr in tag.select('tbody tr'):
        link = tr.select_one('a[href*="/athletes/"]')
        m = re.search(r'/athletes/(\d+)', link.get('href', '')) if link else None
        ids.append(m.group(1) if m else None)
    if len(ids) != len(df):
        print(f'SKIP table {index}: athlete-link rows {len(ids)} != parsed rows {len(df)}')
        return None
    df['athlete_id'] = ids
    df = df[df['athlete_id'].notna() & df['athlete_name'].notna() & df['school'].notna()].copy()
    if df.empty:
        return None
    for optional in ('class_year', 'wind', 'converted_mark'):
        if optional not in df: df[optional] = None
    if 'points' not in df: df['points'] = 0
    df['points'] = pd.to_numeric(df['points'], errors='coerce').fillna(0)
    df['place'] = pd.to_numeric(df['place'], errors='coerce')
    df['year'], df['gender'], df['event'] = year, gender, event
    df['event_type'], df['round'] = event_type, round_name
    df['source_table'], df['source_heading'] = index, heading
    df['mark_source_column'] = mark_col
    return df[['year','gender','event','event_type','round','athlete_id',
               'athlete_name','class_year','school','place','mark','converted_mark',
               'points','wind','source_table','source_heading','mark_source_column']]


def process_championship(path, gender, year, field_tables=None, verified_marks=None):
    if not path.exists():
        raise FileNotFoundError(f'Missing saved {gender} HTML: {path}')
    soup = BeautifulSoup(path.read_text(encoding='utf-8'), 'html.parser')
    # Validate actual results headings, not the file name.
    event_headings = [h.get_text(' ', strip=True) for h in soup.find_all(['h1','h2','h3','h4','h5'])]
    # Anchor to the beginning: "Men's" is a substring of "Women's"!
    expected = r"^Women['’]s\b" if gender == 'Women' else r"^Men['’]s\b"
    opposite = r"^Men['’]s\b" if gender == 'Women' else r"^Women['’]s\b"
    matching = [h for h in event_headings if re.search(expected, h, re.I)]
    conflicting = [h for h in event_headings if re.search(opposite, h, re.I)]
    if not matching or conflicting:
        raise ValueError(
            f"Wrong results HTML for {gender}: {path}. "
            f"Matching headings={len(matching)}, opposite-gender headings={len(conflicting)}. "
            "Save the correct results page and rerun all cells."
        )
    for tag in soup.find_all(attrs={'colspan': True}):
        if not str(tag['colspan']).isdigit():
            tag['colspan'] = '1'
    table_tags = soup.find_all('table')
    tables = pd.read_html(StringIO(str(soup)))
    if len(table_tags) != len(tables):
        raise ValueError('HTML tables and pandas tables are not aligned')
    if field_tables is None:
        field_tables = find_main_field_tables(tables, table_tags)
    expected_fields = {
        'High Jump', 'Pole Vault', 'Long Jump', 'Triple Jump',
        'Shot Put', 'Discus', 'Hammer', 'Javelin'
    }
    missing_fields = expected_fields - set(field_tables)
    if missing_fields:
        print(f'WARNING: {year} {gender} missing field summaries: {sorted(missing_fields)}')
    parsed = []
    for i, (table, tag) in enumerate(zip(tables, table_tags)):
        result = clean_event_table(table, tag, i, gender, year, field_tables, verified_marks or {})
        if result is not None:
            parsed.append(result)
    if not parsed:
        raise ValueError(f'No athlete results extracted from {path}')
    results = pd.concat(parsed, ignore_index=True)
    priority = {'Finals': 4, 'Other': 3, 'Preliminaries': 2, 'Flight': 1, 'Heat': 1}
    results['round_priority'] = results['round'].map(priority).fillna(0)
    results['has_mark'] = results['mark'].notna() & results['mark'].astype(str).str.strip().ne('')
    results = results.sort_values(['round_priority', 'has_mark', 'points', 'source_table'],
                                  ascending=[False, False, False, True])
    results = results.drop_duplicates(subset=['year','gender','event','athlete_id'], keep='first')

    field = results[results['event_type'] == 'field'].copy()
    running = results[results['event_type'] == 'running'].copy()
    if not field.empty:
        expected_tables = field['event'].map(field_tables)
        actual_tables = pd.to_numeric(
            field['source_table'], errors='coerce'
        )
        mismatches = actual_tables.ne(expected_tables)

        if mismatches.any():
            raise ValueError(
                "Some field results came from incorrect summary tables:\n"
                + field.loc[
                    mismatches,
                    ['year', 'gender', 'event', 'source_table']
                ].to_string(index=False)
            )
    return running, field



In [9]:
# These are your saved, manually checked selections. Edit this dictionary to approve more.
# Do NOT approve a column until its first three marks match official TFRRS results.
VERIFIED_FIELD_MARK_COLUMNS = {
    (2021, 'Men', 'Discus'): 'MARK.2',
    (2021, 'Men', 'Long Jump'): 'MARK',
    (2021, 'Men', 'Pole Vault'): 'MARK.1',
    (2021, 'Men', 'Shot Put'): 'MARK.2',
    (2021, 'Men', 'Triple Jump'): 'MARK.2',
    (2021, 'Women', 'Discus'): 'MARK.1',
    (2021, 'Women', 'Hammer'): 'MARK.3',
    (2021, 'Women', 'High Jump'): 'MARK.1',
    (2021, 'Women', 'Javelin'): 'MARK',
    (2021, 'Women', 'Pole Vault'): 'MARK.1',
    (2021, 'Women', 'Triple Jump'): 'MARK',
    (2022, 'Men', 'Discus'): 'MARK.4',
    (2022, 'Men', 'Hammer'): 'MARK.1',
    (2022, 'Men', 'High Jump'): 'MARK.2',
    (2022, 'Men', 'Shot Put'): 'MARK',
    (2022, 'Men', 'Triple Jump'): 'MARK.1',
    (2022, 'Women', 'Discus'): 'MARK.6',
    (2022, 'Women', 'High Jump'): 'MARK',
    (2022, 'Women', 'Javelin'): 'MARK',
    (2022, 'Women', 'Long Jump'): 'MARK.3',
    (2022, 'Women', 'Pole Vault'): 'MARK.3',
    (2022, 'Women', 'Triple Jump'): 'MARK.3',
    (2023, 'Men', 'Discus'): 'MARK.5',
    (2023, 'Men', 'Hammer'): 'MARK.2',
    (2023, 'Men', 'High Jump'): 'MARK.2',
    (2023, 'Men', 'Javelin'): 'MARK.3',
    (2023, 'Men', 'Long Jump'): 'MARK.3',
    (2023, 'Men', 'Pole Vault'): 'MARK',
    (2023, 'Men', 'Triple Jump'): 'MARK',
    (2023, 'Women', 'Discus'): 'MARK.4',
    (2023, 'Women', 'Hammer'): 'MARK',
    (2023, 'Women', 'Javelin'): 'MARK.4',
    (2023, 'Women', 'Long Jump'): 'MARK',
    (2023, 'Women', 'Pole Vault'): 'MARK',
    (2023, 'Women', 'Shot Put'): 'MARK.1',
    (2023, 'Women', 'Triple Jump'): 'MARK',
    (2024, 'Men', 'Hammer'): 'MARK',
    (2024, 'Men', 'High Jump'): 'MARK.3',
    (2024, 'Men', 'Javelin'): 'MARK',
    (2024, 'Men', 'Long Jump'): 'MARK.1',
    (2024, 'Men', 'Shot Put'): 'MARK.3',
    (2024, 'Men', 'Triple Jump'): 'MARK',
    (2024, 'Women', 'Discus'): 'MARK.1',
    (2024, 'Women', 'Hammer'): 'MARK.6',
    (2024, 'Women', 'High Jump'): 'MARK.4',
    (2024, 'Women', 'Javelin'): 'MARK.1',
    (2024, 'Women', 'Long Jump'): 'MARK',
    (2024, 'Women', 'Pole Vault'): 'MARK.5',
    (2024, 'Women', 'Shot Put'): 'MARK.2',
    (2024, 'Women', 'Triple Jump'): 'MARK.1',
    (2025, 'Women', 'Triple Jump'): 'MARK',
    (2025, 'Men', 'Discus'): 'MARK',
    (2025, 'Men', 'Long Jump'): 'MARK.2',
    (2025, 'Men', 'Hammer'): 'MARK.1',
    (2025, 'Men', 'High Jump'): 'MARK.5',
    (2025, 'Men', 'Javelin'): 'MARK',
    (2025, 'Men', 'Shot Put'): 'MARK',
    (2025, 'Men', 'Triple Jump'): 'MARK.1',
    (2025, 'Women', 'Discus'): 'MARK.2',
    (2025, 'Women', 'Hammer'): 'MARK',
    (2025, 'Women', 'High Jump'): 'MARK.1',
    (2025, 'Women', 'Javelin'): 'MARK',
    (2025, 'Women', 'Long Jump'): 'MARK.5',
    (2025, 'Women', 'Pole Vault'): 'MARK',
    (2025, 'Women', 'Shot Put'): 'MARK.1',
    (2026, 'Men', 'Discus'): 'MARK.2',
    (2026, 'Men', 'Hammer'): 'MARK',
    (2026, 'Men', 'High Jump'): 'MARK.2',
    (2026, 'Men', 'Javelin'): 'MARK.4',
    (2026, 'Men', 'Long Jump'): 'MARK',
    (2026, 'Men', 'Pole Vault'): 'MARK.5',
    (2026, 'Men', 'Shot Put'): 'MARK.1',
    (2026, 'Men', 'Triple Jump'): 'MARK.1',
    (2026, 'Women', 'High Jump'): 'MARK.3',
    (2026, 'Women', 'Javelin'): 'MARK.3',
    (2026, 'Women', 'Long Jump'): 'MARK.1',
    (2026, 'Women', 'Pole Vault'): 'MARK',
    (2026, 'Women', 'Shot Put'): 'MARK.3',
    (2026, 'Women', 'Triple Jump'): 'MARK.2',
    (2026, 'Women', 'Discus'): 'MARK',
    (2026, 'Women', 'Hammer'): 'MARK',    
}


In [10]:
# Review candidate MARK columns, showing the first three finishers for each.
# Only field-event summary tables are shown; approved events are hidden by default.
def audit_source(path, gender, year):
    soup = BeautifulSoup(path.read_text(encoding='utf-8'), 'html.parser')
    for tag in soup.find_all(attrs={'colspan': True}):
        if not str(tag['colspan']).isdigit():
            tag['colspan'] = '1'
    tags = soup.find_all('table')
    tables = pd.read_html(StringIO(str(soup)))
    if len(tags) != len(tables):
        raise ValueError(f'Table alignment failed: {path}')
    summaries = find_main_field_tables(tables, tags)
    rows = []
    for event, idx in summaries.items():
        table = tables[idx]
        cols = [c for c in table.columns if re.fullmatch(r'MARK(?:\.\d+)?', str(c))]
        if len(cols) <= 1:
            continue  # No ambiguity to resolve
        sample = table.sort_values('PL', key=lambda s: pd.to_numeric(s, errors='coerce')) if 'PL' in table else table
        for col in cols:
            vals = ['—' if pd.isna(v) else str(v) for v in sample[col].head(3)]
            rows.append({'year': year, 'gender': gender, 'event': event,
                         'candidate_column': col,
                         'first_three_marks': ' | '.join(vals),
                         'approved': VERIFIED_FIELD_MARK_COLUMNS.get((year, gender, event)) == col})
    return rows

available_sources = []
missing_sources = []
for year in YEARS:
    for gender in GENDERS:
        path = RAW_DIR / f'bigwest_{year}_{gender.lower()}.html'
        (available_sources if path.exists() else missing_sources).append((year, gender, path))
if STRICT_INPUTS and missing_sources:
    raise FileNotFoundError('Missing saved HTML: ' + ', '.join(str(p) for _, _, p in missing_sources))
if not available_sources:
    raise FileNotFoundError(f'No saved meet HTML files in {RAW_DIR}')
if missing_sources:
    print(f'Skipping {len(missing_sources)} missing HTML files: ' + ', '.join(f'{y} {g}' for y,g,_ in missing_sources))

rows = [row for year, gender, path in available_sources for row in audit_source(path, gender, year)]
mark_audit = pd.DataFrame(rows, columns=['year','gender','event','candidate_column','first_three_marks','approved'])
mark_audit.to_csv(OUTPUT_DIR / 'bigwest_mark_column_audit.csv', index=False)
# A verified choice is valid only if it matches an actual candidate column.
if mark_audit.empty:
    pending = mark_audit.copy()
else:
    event_keys = ['year', 'gender', 'event']
    approved_events = mark_audit.loc[mark_audit['approved'], event_keys].drop_duplicates()
    pending = mark_audit.merge(
        approved_events.assign(_verified=True),
        on=event_keys, how='left'
    )
    pending = pending.loc[pending['_verified'].isna()].drop(columns='_verified')

remaining = pending[['year', 'gender', 'event']].drop_duplicates()
print(f'{len(remaining)} field events still need verification.')
if not pending.empty:
    display(pending.sort_values(['year', 'gender', 'event', 'candidate_column']))
# For the complete audit, run: display(mark_audit)


0 field events still need verification.


In [11]:
# Block export until every multi-MARK field event has an approved column.
if not pending.empty:
    raise ValueError('Unverified field MARK columns. Compare first_three_marks with TFRRS, add selections in the approvals cell, and rerun from that cell.')

running_results, field_results = [], []
for year, gender, path in available_sources:
    choices = {event: col for (yr, gen, event), col in VERIFIED_FIELD_MARK_COLUMNS.items()
               if (yr, gen) == (year, gender)}
    running, field = process_championship(path, gender, year, verified_marks=choices)
    running_results.append(running)
    field_results.append(field)
    print(f'{year} {gender}: {len(running)} running, {len(field)} field rows')

running_df = pd.concat(running_results, ignore_index=True)
field_df = pd.concat(field_results, ignore_index=True)
if field_df.empty or running_df.empty:
    raise ValueError('No running or field results were extracted; check the saved HTML and parser.')
# Name files according to the years actually included.
years_present = [year for year, _, _ in available_sources]
stem = f'bigwest_{min(years_present)}_{max(years_present)}'
running_path = OUTPUT_DIR / f'{stem}_running.csv'
field_path = OUTPUT_DIR / f'{stem}_field.csv'
running_df.to_csv(running_path, index=False)
field_df.to_csv(field_path, index=False)
print('Saved:', running_path, field_path)


2021 Men: 147 running, 135 field rows
2021 Women: 165 running, 140 field rows
2022 Men: 186 running, 122 field rows
2022 Women: 206 running, 135 field rows
2023 Men: 201 running, 130 field rows
2023 Women: 192 running, 155 field rows
2024 Men: 211 running, 117 field rows
2024 Women: 214 running, 144 field rows
2025 Men: 246 running, 153 field rows
2025 Women: 257 running, 171 field rows
2026 Men: 214 running, 143 field rows
2026 Women: 269 running, 164 field rows
Saved: ..\data\processed\bigwest_2021_2026_running.csv ..\data\processed\bigwest_2021_2026_field.csv


In [15]:
import pandas as pd

running = pd.read_csv("../data/processed/bigwest_2021_2026_running.csv")
field = pd.read_csv("../data/processed/bigwest_2021_2026_field.csv")

athletes = pd.concat([
    running[["athlete_id", "athlete_name"]],
    field[["athlete_id", "athlete_name"]]
]).drop_duplicates(subset=["athlete_id"])

print(f"Unique athletes: {len(athletes)}")

Unique athletes: 1514


In [17]:
import requests

response = requests.get(
    "https://api.parse.bot/scraper/1b66c7cb-769a-4301-82ff-8d308afee7ad/search_athlete",
    headers={"X-API-Key": "pmx_127bea8970ed7f3b4ddaec353e4310e8", "API-Snapshot-Version": "5"},
    params={
        "query": "Emma Coburn"
    },
)

print(response.json())

{'status': 'success', 'data': {'athletes': [{'athlete_id': '2258081', 'name': 'Coburn, Emma', 'url': 'https://tfrrs.org/athletes/2258081/Colorado/Emma_Coburn.html'}]}}
